In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install ultralytics
!pip install supervision

In [ ]:
# 查看已安装的版本
import ultralytics
ultralytics.checks()

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2
from ultralytics import YOLO
import supervision as sv
from IPython.display import Video
import warnings
warnings.filterwarnings('ignore')
# 关闭此 Notebook 的 Weights & Biases (wandb) 报告功能
os.environ['WANDB_DISABLED'] = 'true'

In [ ]:
import os
import random
import cv2
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
# 读取数据 .yaml 文件获取类别名称列表
def read_class_names(data_yaml_path):
    """
    读取数据 .yaml 文件获取类别名称列表。

    参数：
    - data_yaml_path: str，数据 .yaml 文件的路径。

    返回：
    - class_names: list，类别名称列表
    """
    with open(data_yaml_path, 'r') as f:
        lines = f.readlines()
    names_line = next(line for line in lines if line.startswith('names:'))
    class_names = eval(names_line.split(':')[1].strip())
    return class_names

In [ ]:
# 解析 YOLO 格式的 .txt 文件
def parse_yolo_label(label_path, image_width, image_height):
    """
    解析 YOLO 格式的标签文件

    参数：
    - label_path: str，标签文件的路径
    - image_width: int，图像的宽度
    - image_height: int，图像的高度

    返回：
    - boxes: list，包含边界框信息的列表，每个边界框为 [class_id, x_min, y_min, x_max, y_max]
    """
    boxes = []
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            class_id = parts[0]
            x_center, y_center, width, height = map(float, parts[1:])
            x_min = (x_center - width / 2) * image_width
            y_min = (y_center - height / 2) * image_height
            x_max = (x_center + width / 2) * image_width
            y_max = (y_center + height / 2) * image_height
            boxes.append((class_id, x_min, y_min, x_max, y_max))
    return boxes

In [ ]:
# 在图片上绘制边界框和类别名称
def draw_boxes(image, boxes, class_names):
    """
    在图片上绘制边界框和列别名称

    参数：
    - image: ndarray，输入图片
    - boxes: list，包含边界框信息的列表
    - class_names: list，类别名称列表
    """
    for box in boxes:
        class_id, x_min, y_min, x_max, y_max = box
        label = class_names[int(class_id)]
        color = tuple(random.randint(0, 255) for _ in range(3)) # 随机颜色
        cv2.rectangle(image, (int(x_min), int(y_min)), (int(x_max), int(y_max)), color, 2)
        font_scale = 0.7
        thickness = 1
        text_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, font_scale, thickness)[0]
        text_x = int(x_min)
        text_y = int(y_min) - 5 if int(y_min) - 5 > 5 else int(y_min) + 5
        cv2.rectangle(image, (text_x, text_y - text_size[1]), (text_x + text_size[0], text_y), color, -1)
        cv2.putText(image, label, (text_x, text_y), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (255, 255, 255), thickness)

In [ ]:
data_yaml_path = '/kaggle/input/datasets/pkdarabi/cardetection/car/data.yaml'
images_dir = '/kaggle/input/datasets/pkdarabi/cardetection/car/train/images'
labels_dir = '/kaggle/input/datasets/pkdarabi/cardetection/car/train/labels'

In [ ]:
class_names = read_class_names(data_yaml_path)
image_files = [f for f in os.listdir(images_dir) if f.endswith('.jpg')]
selected_images = random.sample(image_files, 2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for i, img_file in enumerate(selected_images):
    img_path = os.path.join(images_dir, img_file)
    label_path = os.path.join(labels_dir, img_file.replace('.jpg', '.txt'))

    image = cv2.imread(img_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image_height, image_width, _ = image.shape

    boxes = parse_yolo_label(label_path, image_width, image_height)
    draw_boxes(image_rgb, boxes, class_names)

    axes[i].imshow(image_rgb)
    axes[i].set_title(img_file)
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## 训练交通标志检测模型

### 模型下载

In [ ]:
model = YOLO('yolo11n.pt')

In [ ]:
# 随机选择 3 张图片
selected_images = random.sample(image_files, 3)

# 创建一个 1 * 3 的子图，用于显示图片，指定图片大小为 16 * 8 英寸（1 英寸 = 0.254 米）
fig, axes = plt.subplots(1, 3, figsize = (16, 8))

# 遍历选中的图片文件
for i, img_file in enumerate(selected_images):
    # 构造图片的完整路径
    img_path = os.path.join(images_dir, img_file)
    # 构造标签的完整路径，假设标签文件名和图片文件名同名，但扩展名为 .txt
    label_path = os.path.join(labels_dir, img_file.replace('.jpg', '.txt'))

    # 读取图片文件
    image = cv2.imread(img_path)
    # 将图片从 BGR 颜色空间转换为 RGB 颜色空间
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # 使用模型对图片进行处理，获取检测结果
    results = model(image_rgb)
    # 提取第一个检测结果
    detection_results = results[0]
    # 从检测结果中构建 detections 对象
    detections = sv.Detections.from_ultralytics(detection_results)
    # 创建一个框注释器对象，用于在图片上绘制检测框
    box_annotator = sv.BoxAnnotator()
    # 在图片副本上绘制检测框
    annotated_image = box_annotator.annotate(
        scene = image.copy(),
        detections = detections
    )
    # 创建一个标签注释器对象
    label_annotator = sv.LabelAnnotator()
    # 根据检测到的类别 ID 获取对应的标签名称
    labels = [
        model.model.names[class_id] for class_id in detections.class_id
    ]
    # 在图片上添加标签
    annotated_image = label_annotator.annotate(
        scene = annotated_image,
        detections = detections,
        labels = labels
    )

    # 在子图中显示注释后的图片
    axes[i].imshow(annotated_image)
    # 设置子图的标题为图片文件名
    axes[i].set_title(img_file)
    # 关闭子图的坐标系显示
    axes[i].axis('off')

### 模型训练

In [ ]:
results = model.train(
    data = '/kaggle/input/datasets/pkdarabi/cardetection/car/data.yaml',
    epochs = 30,
    imgsz = 416,
    batch = 64,
    device = [0, 1]
)

In [ ]:
# 执行性能评估曲线的图片所在的目录
directory = '/kaggle/working/runs/detect/train'

# 定义要展示的图片文件名
image_files = [
    'BoxF1_curve.png',
    'BoxP_curve.png',
    'BoxR_curve.png',
    'BoxPR_curve.png'
]

# 创建一个包含 4 张子图的图形
fig, axes = plt.subplots(2, 2, figsize = (16, 10))
axes = axes.flatten() # 将二维数组展平为一维数组

# 遍历每个图片文件并加载到相应的子图中
for ax, image_file in zip(axes, image_files):
    image_path = os.path.join(directory, image_file)
    if os.path.exists(image_path):
        img = plt.imread(image_path)
        ax.imshow(img)
        ax.set_title(os.path.splitext(image_file)[0]) # 设置子图标题为图片文件名（去除扩展名）
        ax.axis('off') # 关闭坐标轴
    else:
        ax.text(0.5, 0.5, f"File not found: {image_file}", ha = 'center', va = 'enter', fontsize = 12)
        ax.axis('off')

# 调整子图间距
plt.tight_layout()
plt.show()

In [ ]:
# results.png 图片路径
image_path = '/kaggle/working/runs/detect/train/results.png'
# 读取图片
img = plt.imread(image_path)
# 创建一个图形窗口并显示图片
plt.figure(figsize = (12, 8))
plt.imshow(img)
plt.title('Results plot during training')
plt.axis('off')
plt.show()

### 模型验证

In [ ]:
# 加载训练好的最优模型
model = YOLO('/kaggle/working/runs/detect/train/weights/best.pt')
# 模型验证
metrics = model.val(data = '/kaggle/input/datasets/pkdarabi/cardetection/car/data.yaml')

In [ ]:
%matplotlib inline
# 创建柱状图
ax = sns.barplot(
    x = ['mAP50-95', 'mAP50', 'mAP75'],
    y = [metrics.box.map, metrics.box.map50, metrics.box.map75]
)
# 设置标题和轴标签
ax.set_title('YOLO Evaluation Metrics')
ax.set_xlabel('Metric')
ax.set_ylabel('Value')
fig = plt.gcf()
fig.set_size_inches(8, 6)

# 在柱状图上方添加数值
for p in ax.patches:
    ax.annotate(
        '{:.3f}'.format(p.get_height()),
        (p.get_x() + p.get_width() / 2, p.get_height()),
        ha = 'center',
        va = 'center'
    )

# 显示图形
plt.show()

### 模型预测

In [ ]:
test_img_dir = '/kaggle/input/datasets/pkdarabi/cardetection/car/test/images'
num_samples = 9
image_files = os.listdir(test_img_dir)
test_imgs = random.sample(image_files, num_samples)

fig, axes = plt.subplots(3, 3, figsize = (11, 11))
for i in range(num_samples):
    image = test_imgs[i]
    ax = axes[i // 3, i % 3]
    result_predict = model.predict(source = os.path.join(test_img_dir, image), imgsz = (416))
    ax.imshow(result_predict[0].plot())
    ax.set_title(f'Image {i + 1}')
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
!ffmpeg -y -loglevel panic -i /kaggle/input/datasets/pkdarabi/cardetection/video.mp4 output.mp4

In [ ]:
Video('output.mp4', width = 720, embed = True)

In [ ]:
!yolo track model = '/kaggle/working/runs/detect/train/weights/best.pt' source = '/kaggle/input/datasets/pkdarabi/cardetection/video.mp4' show = True

In [ ]:
# 查看检测结果
# 格式转换
!ffmpeg -y -loglevel panic -i /kaggle/working/runs/detect/track/video.avi result_out.mp4

In [ ]:
# 展示视频
Video("result_out.mp4", width = 720)

In [ ]:
model.export(format = 'onnx')

### 将结果导出到本地

In [ ]:
!zip -q -r result.zip /kaggle/working/*